# 第一节 加载公开词向量

In [13]:
from gensim.models import KeyedVectors
from transformers.convert_slow_tokenizer import import_protobuf

model_path='./data/sgns.weibo.word.bz2'
# 读取 word2vec 格式的预训练词向量文件，返回一个`KeyedVectors`对象，拿到词‑向量映射，之后就可以做相似度、类比、取词向量等操作。
model = KeyedVectors.load_word2vec_format(model_path)

In [14]:
# 1.查看词向量维度
print(model.vector_size)

300


In [15]:
# 2.词数
print(len(model.index_to_key))

195202


In [16]:
# 3.查看某个词的向量
print(model['地铁'])

[ 2.92064e-01 -5.18680e-02 -2.13720e-01  1.82131e-01  2.82900e-03
  4.14104e-01  1.56440e-01 -1.27940e-02 -3.28332e-01 -8.25000e-02
 -8.46890e-02 -2.14700e-02  1.18650e-01 -4.73659e-01 -1.97850e-02
  1.13939e-01  1.82734e-01 -6.46420e-02  5.60832e-01 -6.65230e-02
 -1.97960e-01  1.26039e-01 -3.28720e-01 -3.09730e-02 -3.46580e-01
 -1.53190e-01 -2.96226e-01 -5.75517e-01  1.10684e-01  8.19220e-02
 -1.04721e-01 -1.77477e-01 -1.21332e-01  1.49816e-01  2.86278e-01
 -8.11200e-03  6.72540e-02  6.92220e-02 -3.50973e-01 -5.49500e-02
 -7.80250e-02 -1.92952e-01 -1.70920e-01 -1.28289e-01  1.08204e-01
 -7.24913e-01 -1.11735e-01 -6.75000e-03  4.38086e-01 -8.75720e-02
 -1.41320e-01 -1.91726e-01  1.68363e-01 -7.85700e-02 -1.79772e-01
 -1.27950e-01  3.24675e-01  2.70616e-01  1.96330e-02 -3.09431e-01
 -4.02670e-02  5.80160e-02 -1.06603e-01  2.48022e-01  2.59204e-01
  2.16016e-01  3.96293e-01  2.48630e-02 -7.63330e-02  6.10183e-01
  2.73094e-01 -1.37777e-01  7.65180e-02 -1.35788e-01 -1.42440e-01
  1.22921e

In [17]:
# 4.查看两个向量的相似度（余弦相似度）
similarity=model.similarity('地铁','公交')
print('地铁 vs 公交 相似度：',similarity)
similarity=model.similarity('地铁','图书馆')
print('地铁 vs 公交 相似度：',similarity)

地铁 vs 公交 相似度： 0.65458214
地铁 vs 公交 相似度： 0.2721027


In [18]:
# 5.最相似
print(model.most_similar(positive=['地铁']))
print(model.most_similar(positive=['男人','女孩'],negative=['男孩'],topn=5))

[('二号线', 0.6998022198677063), ('四号线', 0.6872348785400391), ('北京地铁', 0.6863653659820557), ('一号线', 0.6666116118431091), ('地铁站', 0.659015417098999), ('公交', 0.654582142829895), ('五号线', 0.6538913249969482), ('坐地铁', 0.6435028314590454), ('军博', 0.6391095519065857), ('八通线', 0.6323882341384888)]
[('女人', 0.6578881740570068), ('女孩子', 0.515068531036377), ('女生', 0.45194485783576965), ('女人真', 0.4420627951622009), ('女人们', 0.43698593974113464)]


# 第二节 训练自己的词向量

In [40]:
import pandas as pd
import jieba
from gensim.models import Word2Vec

In [41]:
df=pd.read_csv('./data/online_shopping_10_cats.csv',encoding='utf-8',).dropna()

In [42]:
df.head()

,cat,label,review
0,书籍,1,做父母一定要有刘墉这样的心态，不断地学习，不断地进步，不断地给自己补充新鲜血液，让自己保持一...
1,书籍,1,作者真有英国人严谨的风格，提出观点、进行论述论证，尽管本人对物理学了解不深，但是仍然能感受到...
2,书籍,1,作者长篇大论借用详细报告数据处理工作和计算结果支持其新观点。为什么荷兰曾经县有欧洲最高的生产...
3,书籍,1,作者在战几时之前用了＂拥抱＂令人叫绝．日本如果没有战败，就有会有美军的占领，没胡官僚主义的延...
4,书籍,1,作者在少年时即喜阅读，能看出他精读了无数经典，因而他有一个庞大的内心世界。他的作品最难能可贵...


In [52]:
sentences=[[token for token in jieba.lcut(sentence) if token.strip() != ''] for sentence in df['review']]

In [53]:
model=Word2Vec(
    sentences,           #已分词的句子序列
    vector_size=100,     #词向量维度
    window=5,            #上下窗口大小
    min_count=2,         #最小词频（低于直接忽略）
    sg=1,                #1:Skip-Gram，0:CBOW
    workers=4,           #并行训练线程数
)

In [54]:
model.wv.save_word2vec_format('./data/word2vec.txt')

In [56]:
KeyedVectors.load_word2vec_format('./data/word2vec.txt')['地铁']

array([ 0.11568071,  0.41075695, -0.32508856,  0.03044724, -0.4062489 ,
       -0.5058771 ,  0.35743025,  0.46679696, -0.7942113 ,  0.18896821,
       -0.35660416, -0.39380094,  0.26691183, -0.08865482,  0.2674271 ,
        0.3831367 ,  0.2844641 , -0.41117302,  0.11406655,  0.0399126 ,
        0.87173015, -0.17216864,  0.13409245, -0.1397665 , -0.3874155 ,
       -0.26306152,  0.07637891,  0.62697023, -0.31690818, -0.15617989,
        0.14168483, -0.0327967 , -0.08293466, -0.19250731,  0.13599464,
        0.08073995,  0.05122366, -0.2751639 ,  0.01126499, -0.4879201 ,
       -0.2655429 , -0.06421432, -0.22638085,  0.13804007,  0.55917674,
       -0.619058  ,  0.36333144,  0.205606  , -0.39720014,  0.70304346,
       -0.17559561, -0.08193552, -0.5182564 ,  0.46929204,  0.17043664,
       -0.49476823,  0.2757367 ,  0.02755577, -0.03813518, -0.3352368 ,
       -0.03078052,  0.41971162,  0.46093923,  0.15875874,  0.39312443,
        0.10133242,  0.45271263,  0.60245943,  0.21746288,  0.39

# 词向量运用

In [62]:
import torch.nn as nn
import torch

In [68]:
# 1.加载词向量
wv=KeyedVectors.load_word2vec_format('./data/word2vec.txt')

In [70]:
# 2.处理OOV
unk_token='<unk>'
index2word=[unk_token]+wv.index_to_key
word2index = {word: idx for idx, word in enumerate(index2word)}

In [71]:
# 3.准备词向量矩阵
num_embeddings=len(index2word)
embedding_dim=wv.vector_size
embedding_martix=torch.randn(num_embeddings, embedding_dim)

for index,word in enumerate(index2word):
    if word in wv:
        embedding_martix[index]=torch.tensor(wv[word])

In [72]:
 # 4.创建Embedding
embedding=nn.Embedding.from_pretrained(embedding_martix)

In [74]:
# 5.测试
text="我喜欢乘坐地铁"
tokens=jieba.lcut(text)
input_ids=[word2index.get(token,word2index[unk_token]) for token in tokens]
input_tensor=torch.tensor(input_ids)
embedding(input_tensor).shape

torch.Size([4, 100])